# Data Visualization with Matplotlib & Seaborn

A beginner-friendly Exploratory Data Analysis (EDA) and Data Visualization notebook using the **Seaborn Tips** dataset.

### Project Objectives
1. Load, inspect, and clean real-world restaurant tipping data.
2. Compute descriptive statistics and examine relationships between bill totals, tips, party sizes, and dining times.
3. Build six clear visualizations (Histogram, Bar Chart, Scatter Plot, Box Plot, Correlation Heatmap, Pair Plot).
4. Assemble a 2×2 executive summary dashboard saved at 150 DPI (`outputs/tips_overview.png`).

## 1. Import Required Libraries
We start by importing Pandas and NumPy for data handling, and Matplotlib and Seaborn for creating charts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure clean chart styling
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#FAFAFC'

outputs_dir = Path('../outputs')
outputs_dir.mkdir(parents=True, exist_ok=True)

## 2. Load the Dataset
We load the built-in `tips` dataset directly from Seaborn and save a local CSV copy inside `dataset/`.

In [ ]:
df = sns.load_dataset('tips')
Path('../dataset').mkdir(parents=True, exist_ok=True)
df.to_csv('../dataset/tips.csv', index=False)

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 3. Data Inspection, Missing Values & Duplicate Checking
Before visualizing data, we check column types, missing values, and exact duplicate records.

In [ ]:
df.info()

print('\nMissing values per column:')
print(df.isnull().sum())

dup_count = df.duplicated().sum()
print(f'\nExact duplicate rows found: {dup_count}')
if dup_count > 0:
    display(df[df.duplicated(keep=False)])

## 4. Data Cleaning & Statistical Summary
We remove the single exact duplicate row so each table observation is counted once, then print summary statistics.

In [ ]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f'Cleaned dataset shape: {df_clean.shape}')
df_clean.describe().round(2)

## 5. Exploratory Data Analysis (Grouped Summaries)
Let's inspect average tip amounts across days of the week and meal times.

In [ ]:
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
print('Tip summary by day:')
display(df_clean.groupby('day', observed=False)['tip'].agg(['count', 'mean', 'median', 'max']).reindex(day_order).round(2))

print('\nAverage metrics by meal time:')
display(df_clean.groupby('time', observed=False)[['total_bill', 'tip', 'size']].mean().round(2))

## 6. Visualization 1 — Histogram (Distribution of Total Bills)
- **What it represents**: Frequency distribution of restaurant bill totals across all 243 cleaned tables, overlaid with a smooth Kernel Density Estimate (KDE) curve.
- **Patterns**: The distribution is right-skewed—most bills fall between $10 and $25, with a smaller tail extending up to $50.81.
- **Relationship**: Because a few high-value bills pull the average upward, the mean ($19.81) sits above the median ($17.81).
- **Key Takeaway**: Most dining tables spend between $13.38 and $24.18 (interquartile range), with a median bill of $17.81.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.histplot(data=df_clean, x='total_bill', bins=20, kde=True, color='#2563EB', edgecolor='white', alpha=0.82, ax=ax)
ax.axvline(df_clean['total_bill'].mean(), color='#DC2626', linestyle='--', linewidth=1.8, label=f"Mean: ${df_clean['total_bill'].mean():.2f}")
ax.axvline(df_clean['total_bill'].median(), color='#059669', linestyle=':', linewidth=2.0, label=f"Median: ${df_clean['total_bill'].median():.2f}")
ax.set_title('Distribution of Total Restaurant Bills')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Number of Tables (Frequency)')
ax.legend()
plt.tight_layout()
fig.savefig(outputs_dir / 'histogram.png', dpi=150)
plt.show()

## 7. Visualization 2 — Bar Chart (Average Tips by Day)
- **What it represents**: Mean tip amount collected per table from Thursday through Sunday.
- **Patterns**: Average tips rise toward the weekend, peaking on Sunday ($3.26) followed by Saturday ($2.99), compared to Thursday ($2.78) and Friday ($2.73).
- **Relationship**: Day of the week is associated with different average tip levels, largely because weekend service consists almost entirely of dinner orders.
- **Key Takeaway**: Sunday tables leave the highest average tip ($3.26), roughly 19% more in dollar terms than Friday tables ($2.73).

In [ ]:
day_palette = ['#2563EB', '#0EA5E9', '#0D9488', '#4F46E5']
fig, ax = plt.subplots(figsize=(9, 5.5))
avg_tips = df_clean.groupby('day', observed=False)['tip'].mean().reindex(day_order)
bars = sns.barplot(x=avg_tips.index, y=avg_tips.values, hue=avg_tips.index, palette=day_palette, legend=False, ax=ax)
for bar, val in zip(bars.patches, avg_tips.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.06, f'${val:.2f}', ha='center', va='bottom', fontweight='semibold')
ax.set_ylim(0, max(avg_tips.values) * 1.18)
ax.set_title('Average Tip Amount by Day of Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Tip ($)')
plt.tight_layout()
fig.savefig(outputs_dir / 'bar_chart.png', dpi=150)
plt.show()

## 8. Visualization 3 — Scatter Plot (Total Bill vs. Tip)
- **What it represents**: Each point represents one dining table's total bill (x-axis) against its tip amount (y-axis), grouped by meal time (`Lunch` vs. `Dinner`).
- **Patterns**: Points trend steadily upward from bottom-left to top-right, while spread increases for bills above $30.
- **Relationship**: There is a clear positive linear relationship between total bill and tip (`y = 0.10x + 0.92`, `r = 0.67`), and Dinner accounts for most high-bill, high-tip observations.
- **Key Takeaway**: Higher total bills are strongly associated with larger dollar tips, though individual tipping variance widens on larger checks.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.scatterplot(data=df_clean, x='total_bill', y='tip', hue='time', style='time', palette={'Lunch': '#0EA5E9', 'Dinner': '#1D4ED8'}, s=65, alpha=0.82, ax=ax)
slope, intercept = np.polyfit(df_clean['total_bill'], df_clean['tip'], 1)
x_vals = np.linspace(df_clean['total_bill'].min(), df_clean['total_bill'].max(), 100)
ax.plot(x_vals, slope * x_vals + intercept, color='#DC2626', linestyle='--', linewidth=1.8, label=f'Linear Fit (y = {slope:.2f}x + {intercept:.2f})')
ax.set_title('Relationship Between Total Bill and Tip')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Tip ($)')
ax.legend(title='Meal Time / Fit')
plt.tight_layout()
fig.savefig(outputs_dir / 'scatter_plot.png', dpi=150)
plt.show()

## 9. Visualization 4 — Box Plot (Tip Distribution Across Days)
- **What it represents**: Median, interquartile range (middle 50%), whiskers, and outlier tips for each day of the week.
- **Patterns**: Saturday shows the widest spread and the highest single outlier ($10.00), while Sunday exhibits the highest median tip ($3.15) with no extreme upper outliers.
- **Relationship**: Comparing medians and IQRs shows that Sunday's higher average is consistent across typical tables rather than driven by a single outlier.
- **Key Takeaway**: Sunday delivers the highest typical tip (median $3.15), whereas Saturday experiences the greatest tip variability and largest single outlier ($10.00).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=df_clean, x='day', y='tip', order=day_order, hue='day', hue_order=day_order, palette=day_palette, legend=False, width=0.55, ax=ax)
ax.set_title('Tip Distribution Across Days of the Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Tip ($)')
plt.tight_layout()
fig.savefig(outputs_dir / 'box_plot.png', dpi=150)
plt.show()

## 10. Visualization 5 — Correlation Heatmap
- **What it represents**: Pairwise Pearson correlation coefficients (`r`) among the three numerical features: `total_bill`, `tip`, and `size`.
- **Patterns**: All three pairs show positive correlations, led by `total_bill` and `tip` (`r = 0.67`), followed by `total_bill` and `size` (`r = 0.60`), and `tip` and `size` (`r = 0.49`).
- **Relationship**: Larger parties tend to generate larger total bills, which in turn correspond to higher dollar tips.
- **Key Takeaway**: `total_bill` is a stronger linear correlate of `tip` (`r = 0.67`) than party `size` alone (`r = 0.49`).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
corr = df_clean[['total_bill', 'tip', 'size']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=1.2, ax=ax)
ax.set_title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
fig.savefig(outputs_dir / 'correlation_heatmap.png', dpi=150)
plt.show()

## 11. Visualization 6 — Pair Plot
- **What it represents**: A matrix of scatter plots and diagonal histograms comparing every pair of numerical variables (`total_bill`, `tip`, `size`), colored by meal time.
- **Patterns**: Dinner observations (dark blue) stretch further into higher `total_bill` and `tip` ranges than Lunch observations (light blue), and larger party sizes (`size` = 4, 5, 6) shift the bill distributions rightward.
- **Relationship**: Multi-feature view confirms positive pairwise trends across all three numerical variables without hidden non-linear clusters.
- **Key Takeaway**: Meal time separates dining scale cleanly—Dinner services involve larger bills and slightly larger party sizes than Lunch services.

In [ ]:
pair_grid = sns.pairplot(data=df_clean, vars=['total_bill', 'tip', 'size'], hue='time', palette={'Lunch': '#0EA5E9', 'Dinner': '#1D4ED8'}, diag_kind='hist', height=2.8)
pair_grid.figure.suptitle('Pairwise Relationships Among Numerical Features by Meal Time', y=1.02, fontweight='bold')
pair_grid.savefig(outputs_dir / 'pair_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Four-Chart Executive Dashboard (`tips_overview.png`)
Finally, we combine the Histogram, Box Plot, Scatter Plot, and Correlation Heatmap into a 2×2 subplot dashboard using `plt.tight_layout()` and save it at 150 DPI.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df_clean, x='total_bill', bins=20, kde=True, color='#2563EB', edgecolor='white', alpha=0.82, ax=axes[0, 0])
axes[0, 0].axvline(df_clean['total_bill'].mean(), color='#DC2626', linestyle='--', linewidth=1.6, label=f"Mean: ${df_clean['total_bill'].mean():.2f}")
axes[0, 0].set_title('1. Total Bill Distribution')
axes[0, 0].legend()

sns.boxplot(data=df_clean, x='day', y='tip', order=day_order, hue='day', hue_order=day_order, palette=day_palette, legend=False, width=0.55, ax=axes[0, 1])
axes[0, 1].set_title('2. Tip Distribution by Day')

sns.scatterplot(data=df_clean, x='total_bill', y='tip', hue='time', palette={'Lunch': '#0EA5E9', 'Dinner': '#1D4ED8'}, alpha=0.8, s=55, ax=axes[1, 0])
axes[1, 0].plot(x_vals, slope * x_vals + intercept, color='#DC2626', linestyle='--', linewidth=1.6, label='Trend Line')
axes[1, 0].set_title('3. Total Bill vs. Tip Relationship')
axes[1, 0].legend(title='Meal Time')

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=1, ax=axes[1, 1])
axes[1, 1].set_title('4. Numerical Feature Correlation')

fig.suptitle('Restaurant Tips Dataset — Exploratory Dashboard Overview', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(outputs_dir / 'tips_overview.png', dpi=150)
plt.show()